In [10]:
!pip install -q pmdarima prophet statsmodels langgraph rich

In [11]:
import warnings
import logging

import pandas as pd
import numpy as np

from math import sqrt
from typing import TypedDict, List, Dict, Optional

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from sklearn.model_selection import (
    ParameterGrid,
    TimeSeriesSplit
)

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from prophet import Prophet

from pmdarima import auto_arima

from langgraph.graph import (
    StateGraph,
    END
)

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import box


pmdarima is a Python library used for Automatic ARIMA model selection in Time Series Forecasting.

Instead of manually trying different ARIMA parameters like (1,1,1), (2,1,2), (3,1,1), etc., pmdarima automatically finds the best combination.

====



Imagine you are building a weather prediction system:

Pandas → Reads and organizes the weather data.
NumPy → Performs calculations.
ARIMA / SARIMAX / Prophet → Predict tomorrow's weather.
MAE / RMSE → Check how accurate the prediction is.
LangGraph → Coordinates multiple AI agents, each handling a different part of the forecasting process.
Rich → Displays the results in attractive tables and panels in the terminal.

Together, these imports provide everything needed to build a multi-agent time series forecasting pipeline that loads data, trains forecasting models, evaluates them, and presents the results clearly.

      CSV Dataset
            │
            ▼
      Pandas (Load & Clean Data)
            │
            ▼
      NumPy (Numerical Operations)
            │
            ▼
      TimeSeriesSplit (Train/Test Split)
            │
            ▼
      Forecasting Models
      ├── ARIMA
      ├── SARIMAX
      ├── Prophet
      └── Auto ARIMA
            │
            ▼
      Evaluation Metrics
      ├── MAE
      └── RMSE (using sqrt(MSE))
            │
            ▼
      LangGraph Multi-Agent Workflow
            │
            ▼
      Rich Console Output

| Import                | Purpose                                       | Used For                                    |
| --------------------- | --------------------------------------------- | ------------------------------------------- |
| `warnings`            | Suppresses warning messages                   | Cleaner notebook output                     |
| `logging`             | Records logs and execution details            | Debugging and monitoring                    |
| `pandas as pd`        | Data manipulation library                     | Reading CSV, preprocessing time series data |
| `numpy as np`         | Numerical computations                        | Mathematical operations, arrays             |
| `sqrt`                | Square root function                          | Calculating RMSE                            |
| `TypedDict`           | Defines LangGraph state schema                | State management                            |
| `List`                | Type hint for lists                           | Function annotations                        |
| `Dict`                | Type hint for dictionaries                    | Function annotations                        |
| `Optional`            | Allows optional variables                     | State definitions                           |
| `mean_absolute_error` | Calculates MAE                                | Forecast evaluation                         |
| `mean_squared_error`  | Calculates MSE                                | Forecast evaluation                         |
| `ParameterGrid`       | Generates parameter combinations              | Hyperparameter tuning                       |
| `TimeSeriesSplit`     | Splits time series data                       | Train/Test validation                       |
| `ARIMA`               | ARIMA forecasting model                       | Time series forecasting                     |
| `SARIMAX`             | Seasonal ARIMA model                          | Seasonal forecasting                        |
| `Prophet`             | Facebook/Meta forecasting model               | Trend and seasonality forecasting           |
| `auto_arima`          | Automatically finds the best ARIMA parameters | Model selection                             |
| `StateGraph`          | Creates LangGraph workflow                    | Multi-agent orchestration                   |
| `END`                 | Marks the end of the LangGraph                | Workflow termination                        |
| `Console`             | Rich console output                           | Colored terminal output                     |
| `Table`               | Rich table display                            | Showing datasets and metrics                |
| `Panel`               | Rich panel display                            | Displaying messages                         |
| `box`                 | Table border styles                           | Better table formatting                     |


In [12]:


# =========================================================
# CONFIGURATION
# =========================================================

DATASET_PATH = "Final_Dataset.csv"

TARGET_COLUMN = "quantity_sold"

FORECAST_DAYS = 7


# =========================================================
# LOGGING
# =========================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

warnings.filterwarnings("ignore")

console = Console()

      Configuration
            │
            ▼
      Load Final_Dataset.csv
            │
            ▼
      Select quantity_sold
            │
            ▼
      Forecast Next 7 Days
            │
            ▼
      Logging Records Every Step
            │
            ▼
      Rich Console Displays Results

DATASET_PATH → Tells the computer which notebook (CSV file) to open.

TARGET_COLUMN → Tells the computer what to study (for example, daily sales).

FORECAST_DAYS → Tells the computer how many days into the future to predict.

Logging → Like a diary that writes down everything the computer is doing.

Warnings → Small reminders that are hidden so the screen stays clean.

Rich Console → Makes the results look colorful and easy to read instead of plain text.

Configuration Section

| Variable        | Value                 | Purpose                                    | Overall Use                                         |
| --------------- | --------------------- | ------------------------------------------ | --------------------------------------------------- |
| `DATASET_PATH`  | `"Final_Dataset.csv"` | Specifies the location of the dataset file | Reads the cleaned dataset for forecasting           |
| `TARGET_COLUMN` | `"quantity_sold"`     | Specifies the column to predict            | Used by forecasting models (ARIMA, SARIMA, Prophet) |
| `FORECAST_DAYS` | `7`                   | Number of future days to forecast          | Generates predictions for the next 7 days           |


========

Logging Section

| Code                                                       | Purpose                                 | Overall Use                                            |
| ---------------------------------------------------------- | --------------------------------------- | ------------------------------------------------------ |
| `logging.basicConfig()`                                    | Configures the logging system           | Controls how log messages are displayed                |
| `level=logging.INFO`                                       | Displays INFO level and above           | Shows important execution messages                     |
| `format="%(asctime)s - %(levelname)s - %(message)s"`       | Defines the log message format          | Includes timestamp, log level, and message             |
| `logger = logging.getLogger(__name__)`                     | Creates a logger for the current module | Records events during program execution                |
| `logging.getLogger("cmdstanpy").setLevel(logging.WARNING)` | Reduces Prophet's internal logging      | Hides unnecessary messages from `cmdstanpy`            |
| `warnings.filterwarnings("ignore")`                        | Suppresses warning messages             | Keeps the notebook output clean                        |
| `console = Console()`                                      | Creates a Rich Console object           | Displays colorful tables, panels, and formatted output |


=========

Overall Workflow

| Step | Code                        | Purpose                           |
| ---- | --------------------------- | --------------------------------- |
| 1    | `DATASET_PATH`              | Identify which CSV file to load   |
| 2    | `TARGET_COLUMN`             | Identify which column to forecast |
| 3    | `FORECAST_DAYS`             | Specify the prediction horizon    |
| 4    | `logging.basicConfig()`     | Configure logging behavior        |
| 5    | `logger`                    | Record execution progress         |
| 6    | `warnings.filterwarnings()` | Hide warning messages             |
| 7    | `Console()`                 | Display rich-formatted output     |


==========

Code Explanation

| Code                        | Simple Explanation                                |
| --------------------------- | ------------------------------------------------- |
| `DATASET_PATH`              | Tells Python which dataset to use.                |
| `TARGET_COLUMN`             | Tells the model which column to predict.          |
| `FORECAST_DAYS`             | Tells the model how many future days to forecast. |
| `logging.basicConfig()`     | Sets the logging rules.                           |
| `logger`                    | Writes execution information to the console.      |
| `warnings.filterwarnings()` | Hides warning messages.                           |
| `Console()`                 | Makes the output colorful and easy to read.       |


=======

Simple Example

| Date       | Quantity Sold |
| ---------- | ------------: |
| 01-01-2024 |            52 |
| 02-01-2024 |            49 |
| 03-01-2024 |            55 |
| ...        |           ... |

========

Logging Output Example

2026-08-06 17:30:01 - INFO - Dataset Loaded Successfully

2026-08-06 17:30:02 - INFO - Training ARIMA Model

2026-08-06 17:30:04 - INFO - Forecast Completed Successfully

Each log entry contains:

Timestamp – When the event occurred.
Log Level – INFO, WARNING, or ERROR.
Message – Description of the event.

========

Overall Purpose

| Component      | Role                                                                            |
| -------------- | ------------------------------------------------------------------------------- |
| Configuration  | Stores important project settings (dataset path, target column, forecast days). |
| Logging        | Tracks execution progress and errors in a structured format.                    |
| Warning Filter | Reduces unnecessary warning messages.                                           |
| Rich Console   | Produces professional, colorful terminal output.                                |


========

Overall Flow

Configuration
      │
      ▼
Load Final_Dataset.csv
      │
      ▼
Select quantity_sold
      │
      ▼
Forecast Next 7 Days
      │
      ▼
Logging Records Every Step
      │
      ▼
Rich Console Displays Results

========

In [13]:
import os

print(os.listdir())

['.config', '.ipynb_checkpoints', 'Final_Dataset.csv', 'sample_data']


In [14]:

# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv(DATASET_PATH)

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values("date")

df.set_index("date", inplace=True)

df = df.asfreq("D")


# =========================================================
# FEATURE ENGINEERING
# =========================================================

df["lag1"] = df[TARGET_COLUMN].shift(1)

df["lag7"] = df[TARGET_COLUMN].shift(7)

df = df.dropna()


# =========================================================
# TRAIN TEST SPLIT
# =========================================================

split_index = int(len(df) * 0.8)

train = df.iloc[:split_index]

test = df.iloc[split_index:]

y_train = train[TARGET_COLUMN]

y_test = test[TARGET_COLUMN]


      Final_Dataset.csv
              │
              ▼
      Read CSV
              │
              ▼
      Convert Date Format
              │
              ▼
      Sort by Date
              │
              ▼
      Set Date as Index
              │
              ▼
      Set Daily Frequency
              │
              ▼
      Create lag1 Feature
              │
              ▼
      Create lag7 Feature
              │
              ▼
      Remove Missing Values
              │
              ▼
      Split into Train (80%)
              │
              ▼
      Split into Test (20%)
              │
              ▼
      Ready for Forecasting Models

| Section             | Purpose                                                 |
| ------------------- | ------------------------------------------------------- |
| Load Dataset        | Reads and prepares the dataset                          |
| Date Conversion     | Converts dates into a format suitable for time series   |
| Sorting             | Ensures chronological order                             |
| Date Index          | Makes the DataFrame time-indexed                        |
| Daily Frequency     | Ensures continuous daily observations                   |
| Feature Engineering | Creates lag features (`lag1`, `lag7`)                   |
| Drop Missing Values | Removes rows with missing lag values                    |
| Train-Test Split    | Divides data into training (80%) and testing (20%) sets |


Train-Test Split

| Code                               | Purpose                             | Output              |
| ---------------------------------- | ----------------------------------- | ------------------- |
| `split_index = int(len(df) * 0.8)` | Calculates the 80% split point      | Index for splitting |
| `train = df.iloc[:split_index]`    | Selects the first 80% of data       | Training dataset    |
| `test = df.iloc[split_index:]`     | Selects the remaining 20%           | Testing dataset     |
| `y_train = train[TARGET_COLUMN]`   | Extracts target values for training | Training labels     |
| `y_test = test[TARGET_COLUMN]`     | Extracts target values for testing  | Testing labels      |


====

| Dataset  | Percentage | Rows |
| -------- | ---------: | ---: |
| Training |        80% |   80 |
| Testing  |        20% |   20 |


========

| Feature | Meaning           | Example                             |
| ------- | ----------------- | ----------------------------------- |
| `lag1`  | Yesterday's value | Yesterday's sales                   |
| `lag7`  | Last week's value | Sales on the same weekday last week |


========

| Step                    | Simple Explanation                                                                                              | Overall Purpose                                               |
| ----------------------- | --------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------- |
| **Read CSV**            | Open your notebook with daily sales data.                                                                       | Load the dataset into Python for analysis.                    |
| **Convert Date**        | Make sure every date is understood as a calendar date.                                                          | Enables time series analysis and forecasting.                 |
| **Sort**                | Arrange the days from oldest to newest.                                                                         | Ensures the model learns from data in chronological order.    |
| **lag1**                | Remember how many ice creams you sold yesterday.                                                                | Uses the previous day's sales as a feature for prediction.    |
| **lag7**                | Remember how many you sold on the same day last week.                                                           | Captures weekly patterns in the data.                         |
| **Drop Missing Values** | Remove the first few rows that don't have enough past data to create lag features.                              | Keeps only complete and valid records for training.           |
| **Train-Test Split**    | Let the model study 80% of the days, then test it on the remaining 20% to see how well it predicts unseen data. | Evaluates how accurately the model predicts new, unseen data. |


In [15]:
# =========================================================
# METRICS
# =========================================================

def evaluate(actual, pred):

    actual = np.array(actual)

    pred = np.array(pred)

    mae = mean_absolute_error(actual, pred)

    rmse = sqrt(mean_squared_error(actual, pred))

    mape = np.mean(
        np.abs((actual - pred) / (actual + 1e-10))
    ) * 100

    return {
        "MAE": round(float(mae), 3),
        "RMSE": round(float(rmse), 3),
        "MAPE": round(float(mape), 3)
    }



| Code                                                               | Purpose                                      | Overall Use                                                   |
| ------------------------------------------------------------------ | -------------------------------------------- | ------------------------------------------------------------- |
| `def evaluate(actual, pred):`                                      | Defines the evaluation function              | Calculates forecasting accuracy                               |
| `actual = np.array(actual)`                                        | Converts actual values into a NumPy array    | Enables numerical calculations                                |
| `pred = np.array(pred)`                                            | Converts predicted values into a NumPy array | Enables numerical calculations                                |
| `mae = mean_absolute_error(actual, pred)`                          | Calculates Mean Absolute Error               | Measures average prediction error                             |
| `rmse = sqrt(mean_squared_error(actual, pred))`                    | Calculates Root Mean Squared Error           | Measures prediction error, giving more weight to large errors |
| `mape = np.mean(np.abs((actual - pred) / (actual + 1e-10))) * 100` | Calculates Mean Absolute Percentage Error    | Measures prediction error as a percentage                     |
| `return {...}`                                                     | Returns all metrics as a dictionary          | Makes results easy to display and compare                     |


==========

Metrics Used

| Metric   | Full Form                      | Formula                            | Meaning                                        |        |                                                      |
| -------- | ------------------------------ | ---------------------------------- | ---------------------------------------------- | ------ | ---------------------------------------------------- |
| **MAE**  | Mean Absolute Error            | Mean of `                          | Actual − Predicted                             | `      | Average prediction error                             |
| **RMSE** | Root Mean Squared Error        | √(Mean of `(Actual − Predicted)²`) | Penalizes large prediction errors more heavily |        |                                                      |
| **MAPE** | Mean Absolute Percentage Error | Mean of `                          | (Actual − Predicted) / Actual                  | × 100` | Percentage error between actual and predicted values |


====



        Overall workflow
        ----------------

        Actual Values
              │
              ▼
        Predicted Values
              │
              ▼
        Convert to NumPy Arrays
              │
              ▼
        Calculate MAE
              │
              ▼
        Calculate RMSE
              │
              ▼
        Calculate MAPE
              │
              ▼
        Return Dictionary

| Metric   | Best For                                                                                          |
| -------- | ------------------------------------------------------------------------------------------------- |
| **MAE**  | Understanding the average prediction error                                                        |
| **RMSE** | Detecting and penalizing large forecasting errors                                                 |
| **MAPE** | Measuring prediction error as a percentage, making it easier to interpret across different scales |


MAE → "On average, how many marks was your guess off?"
RMSE → "Did you make any very large mistakes? Those count more."
MAPE → "How wrong were your guesses in percentage terms?"

These three scores help you understand how good your predictions are. Lower values generally mean the forecasting model is making more accurate predictions.

In [16]:
# =========================================================
# STATE
# =========================================================

class ForecastState(TypedDict):

    models: List[str]

    results: List[Dict]

    best_model: str

    tuning_method: str

    best_params: Dict

    forecast_df: pd.DataFrame

    error: Optional[str]



| Variable        | Data Type       | Purpose                                                                       | Used By                     |
| --------------- | --------------- | ----------------------------------------------------------------------------- | --------------------------- |
| `models`        | `List[str]`     | Stores the names of forecasting models to run (e.g., ARIMA, SARIMA, Prophet). | Planner Agent, Model Agents |
| `results`       | `List[Dict]`    | Stores evaluation metrics (MAE, RMSE, MAPE) for each model.                   | Evaluation Agent            |
| `best_model`    | `str`           | Stores the name of the best-performing model.                                 | Model Selection Agent       |
| `tuning_method` | `str`           | Stores the hyperparameter tuning method used (Grid Search, Auto ARIMA, etc.). | Tuning Agent                |
| `best_params`   | `Dict`          | Stores the best parameters found during tuning.                               | Tuning Agent                |
| `forecast_df`   | `pd.DataFrame`  | Stores the final forecasted values.                                           | Forecast Agent              |
| `error`         | `Optional[str]` | Stores any error message if a model fails.                                    | Debugger/Error Handler      |


=====

| Component         | Description                                                                                                |
| ----------------- | ---------------------------------------------------------------------------------------------------------- |
| **ForecastState** | A `TypedDict` that stores all information shared between LangGraph agents during the forecasting workflow. |
| **Purpose**       | Acts as the **memory** of the multi-agent system. Each agent reads from and writes to this shared state.   |


======

| Step | State Variable  | Purpose                           |
| ---- | --------------- | --------------------------------- |
| 1    | `models`        | Store models to evaluate          |
| 2    | `results`       | Save evaluation metrics           |
| 3    | `best_model`    | Select the best forecasting model |
| 4    | `tuning_method` | Record how parameters were tuned  |
| 5    | `best_params`   | Save optimal model parameters     |
| 6    | `forecast_df`   | Store forecast results            |
| 7    | `error`         | Store any execution errors        |


=======



      Data flow
      ----------


      ForecastState
            │
            ▼
      Models List
            │
            ▼
      Run Forecast Models
            │
            ▼
      Store Results
            │
            ▼
      Select Best Model
            │
            ▼
      Store Best Parameters
            │
            ▼
      Generate Forecast
            │
            ▼
      Save Forecast DataFrame
            │
            ▼
      Store Errors (if any)

| Without ForecastState                           | With ForecastState                                      |
| ----------------------------------------------- | ------------------------------------------------------- |
| Agents cannot share information easily.         | All agents share one common state.                      |
| Data must be passed manually between functions. | LangGraph automatically passes the updated state.       |
| Harder to coordinate multiple agents.           | Easy collaboration between agents.                      |
| Difficult to track results and errors.          | Results, forecasts, and errors are stored in one place. |


In [17]:
# =========================================================
# MODEL TRAINERS
# =========================================================

def run_arima():

    model = ARIMA(y_train, order=(1, 1, 1))

    fit = model.fit()

    pred = fit.forecast(len(y_test))

    metrics = evaluate(y_test, pred)

    return metrics


def run_sarimax():

    model = SARIMAX(y_train, order=(1, 1, 1))

    fit = model.fit()

    pred = fit.forecast(len(y_test))

    metrics = evaluate(y_test, pred)

    return metrics


def run_prophet():

    prophet_df = df.reset_index()[["date", TARGET_COLUMN]]

    prophet_df.columns = ["ds", "y"]

    train_p = prophet_df.iloc[:split_index]

    test_p = prophet_df.iloc[split_index:]

    model = Prophet()

    model.fit(train_p)

    future = model.make_future_dataframe(
        periods=len(test_p)
    )

    forecast = model.predict(future)

    pred = forecast["yhat"].iloc[-len(test_p):]

    metrics = evaluate(test_p["y"], pred)

    return metrics



Overall Purpose

| Function        | Purpose                            | Model Used | Output          |
| --------------- | ---------------------------------- | ---------- | --------------- |
| `run_arima()`   | Trains an ARIMA forecasting model  | ARIMA      | MAE, RMSE, MAPE |
| `run_sarimax()` | Trains a SARIMAX forecasting model | SARIMAX    | MAE, RMSE, MAPE |
| `run_prophet()` | Trains a Prophet forecasting model | Prophet    | MAE, RMSE, MAPE |


=========

| Code                               | Purpose                             | Output                |
| ---------------------------------- | ----------------------------------- | --------------------- |
| `ARIMA(y_train, order=(1,1,1))`    | Creates an ARIMA model              | ARIMA model           |
| `fit = model.fit()`                | Trains the model                    | Trained model         |
| `pred = fit.forecast(len(y_test))` | Predicts values for the test period | Forecast values       |
| `metrics = evaluate(y_test, pred)` | Evaluates prediction accuracy       | MAE, RMSE, MAPE       |
| `return metrics`                   | Returns the evaluation metrics      | Dictionary of metrics |


=======

| Code                               | Purpose                    | Output           |
| ---------------------------------- | -------------------------- | ---------------- |
| `SARIMAX(y_train, order=(1,1,1))`  | Creates a SARIMAX model    | SARIMAX model    |
| `fit = model.fit()`                | Trains the model           | Trained model    |
| `pred = fit.forecast(len(y_test))` | Forecasts future values    | Predictions      |
| `metrics = evaluate(y_test, pred)` | Calculates MAE, RMSE, MAPE | Accuracy metrics |
| `return metrics`                   | Returns evaluation metrics | Dictionary       |


=====

run_prophet() Explanation
=========================

| Code                                        | Purpose                                           | Output             |
| ------------------------------------------- | ------------------------------------------------- | ------------------ |
| `df.reset_index()`                          | Converts the date index back into a normal column | DataFrame          |
| `["date", TARGET_COLUMN]`                   | Selects only the date and target columns          | Prophet input      |
| `columns = ["ds", "y"]`                     | Renames columns to Prophet's required format      | Prophet-ready data |
| `train_p`                                   | Training data                                     | Training DataFrame |
| `test_p`                                    | Testing data                                      | Testing DataFrame  |
| `model = Prophet()`                         | Creates a Prophet model                           | Prophet model      |
| `model.fit(train_p)`                        | Trains the model                                  | Trained model      |
| `future = model.make_future_dataframe(...)` | Creates future dates                              | Future DataFrame   |
| `forecast = model.predict(future)`          | Predicts future values                            | Forecast DataFrame |
| `pred = forecast["yhat"]`                   | Extracts predicted values                         | Predictions        |
| `metrics = evaluate(test_p["y"], pred)`     | Calculates forecasting accuracy                   | MAE, RMSE, MAPE    |
| `return metrics`                            | Returns evaluation metrics                        | Dictionary         |


      Training Data
            │
            ▼
      ARIMA Model
            │
            ▼
      Train Model
            │
            ▼
      Predict Test Data
            │
            ▼
      Evaluate Accuracy
            │
            ▼
      Return Metrics

      Overall workflow

      Training Dataset
              │
              ▼
      Select Forecast Model
              │
      ┌──────┼─────────┐
      ▼      ▼         ▼
      ARIMA SARIMAX Prophet
      │       │         │
      ▼       ▼         ▼
      Train Model
      │       │         │
      ▼       ▼         ▼
      Predict Test Data
      │       │         │
      ▼       ▼         ▼
      Evaluate Accuracy
      │       │         │
      └──────┼─────────┘
              ▼
      Return Metrics

| Student     | Method                                                               |
| ----------- | -------------------------------------------------------------------- |
| **ARIMA**   | Looks at previous sales to make a prediction.                        |
| **SARIMAX** | Looks at previous sales and repeating patterns (like weekly trends). |
| **Prophet** | Looks at trends and seasonality to make a prediction.                |


In [18]:
# =========================================================
# AGENT 1 → PLANNER
# =========================================================

def planner_agent(state: ForecastState):

    console.print(
        "\n[bold cyan]Planner Agent Running[/bold cyan]"
    )

    models = [
        "ARIMA",
        "SARIMAX",
        "Prophet"
    ]

    if len(df) > 500:
        models.append("LSTM")

    if len(df) > 2000:
        models.append("TCN")

    console.print(
        f"Selected Models → {models}"
    )

    return {
        "models": models
    }



| Agent             | Purpose                                                                       | Input           | Output                     |
| ----------------- | ----------------------------------------------------------------------------- | --------------- | -------------------------- |
| **Planner Agent** | Decides which forecasting models should be executed based on the dataset size | `ForecastState` | List of forecasting models |


====

| Code                                       | Purpose                                           | Overall Use                                                |
| ------------------------------------------ | ------------------------------------------------- | ---------------------------------------------------------- |
| `def planner_agent(state: ForecastState):` | Defines the Planner Agent function                | Starts the planning process                                |
| `console.print(...)`                       | Displays "Planner Agent Running"                  | Shows the current workflow stage                           |
| `models = ["ARIMA", "SARIMAX", "Prophet"]` | Initializes the default forecasting models        | Models suitable for most datasets                          |
| `if len(df) > 500:`                        | Checks if the dataset has more than 500 records   | Decides whether to include a deep learning model           |
| `models.append("LSTM")`                    | Adds the LSTM model                               | Uses LSTM for larger datasets                              |
| `if len(df) > 2000:`                       | Checks if the dataset has more than 2000 records  | Decides whether to include an advanced deep learning model |
| `models.append("TCN")`                     | Adds the Temporal Convolutional Network (TCN)     | Uses TCN for very large datasets                           |
| `console.print(...)`                       | Displays the selected models                      | Lets the user know which models will run                   |
| `return {"models": models}`                | Stores the selected models in the LangGraph state | Passes the model list to the next agent                    |


====

Decision Logic
--------------

| Dataset Size      | Models Selected                    |
| ----------------- | ---------------------------------- |
| **≤ 500 rows**    | ARIMA, SARIMAX, Prophet            |
| **501–2000 rows** | ARIMA, SARIMAX, Prophet, LSTM      |
| **> 2000 rows**   | ARIMA, SARIMAX, Prophet, LSTM, TCN |


=====



    Workflow

    Dataset
        │
        ▼
    Planner Agent
        │
        ▼
    Check Dataset Size
        │
        ├───────────────┐
        │               │
        ▼               ▼
    ≤ 500 Rows      > 500 Rows
        │               │
        ▼               ▼
    ARIMA        ARIMA + LSTM
    SARIMAX      SARIMAX
    Prophet      Prophet
                        │
                        ▼
                  > 2000 Rows?
                        │
                    Yes ─┘
                        ▼
                  Add TCN

The Planner Agent acts like that coach:

It first looks at the size of the dataset.
Then it chooses the most suitable forecasting models.
Finally, it passes the selected models to the next agent for training.

So, the Planner Agent's main job is planning, not forecasting. It makes sure the right models are chosen before any training begins.

In [19]:

# =========================================================
# AGENT 2 → EXECUTOR
# =========================================================

def executor_agent(state: ForecastState):

    console.print(
        "\n[bold yellow]Executor Agent Running[/bold yellow]"
    )

    results = []

    for model_name in state["models"]:

        try:

            if model_name == "ARIMA":

                metrics = run_arima()

            elif model_name == "SARIMAX":

                metrics = run_sarimax()

            elif model_name == "Prophet":

                metrics = run_prophet()

            else:
                continue

            result = {
                "Model": model_name,
                **metrics
            }

            results.append(result)

            console.print(
                f"[green]{model_name} Completed[/green]"
            )

        except Exception as e:

            logger.exception(e)

    return {
        "results": results
    }



| Agent              | Purpose                                                                                              | Input                               | Output                                                       |
| ------------------ | ---------------------------------------------------------------------------------------------------- | ----------------------------------- | ------------------------------------------------------------ |
| **Executor Agent** | Executes all forecasting models selected by the Planner Agent and collects their evaluation metrics. | List of models from `ForecastState` | Results containing model names and metrics (MAE, RMSE, MAPE) |


========

| Code                                        | Purpose                                  | Overall Use                                |
| ------------------------------------------- | ---------------------------------------- | ------------------------------------------ |
| `def executor_agent(state: ForecastState):` | Defines the Executor Agent               | Executes forecasting models                |
| `console.print(...)`                        | Displays "Executor Agent Running"        | Shows the current workflow stage           |
| `results = []`                              | Creates an empty list                    | Stores results from each forecasting model |
| `for model_name in state["models"]:`        | Loops through all selected models        | Executes one model at a time               |
| `if model_name == "ARIMA":`                 | Checks if the current model is ARIMA     | Calls the ARIMA training function          |
| `metrics = run_arima()`                     | Runs the ARIMA model                     | Returns MAE, RMSE, MAPE                    |
| `elif model_name == "SARIMAX":`             | Checks for SARIMAX                       | Calls the SARIMAX training function        |
| `metrics = run_sarimax()`                   | Runs the SARIMAX model                   | Returns evaluation metrics                 |
| `elif model_name == "Prophet":`             | Checks for Prophet                       | Calls the Prophet training function        |
| `metrics = run_prophet()`                   | Runs the Prophet model                   | Returns evaluation metrics                 |
| `else: continue`                            | Skips unsupported models                 | Prevents execution errors                  |
| `result = {"Model": model_name, **metrics}` | Combines the model name with its metrics | Creates one result record                  |
| `results.append(result)`                    | Adds the result to the results list      | Collects all model results                 |
| `console.print(...)`                        | Prints completion message                | Indicates successful execution             |
| `except Exception as e:`                    | Handles execution errors                 | Prevents the workflow from stopping        |
| `logger.exception(e)`                       | Logs the error details                   | Helps with debugging                       |
| `return {"results": results}`               | Stores all results in `ForecastState`    | Passes them to the next agent              |


=======

Imagine a teacher asks three students to solve the same math problem

| Student   | Task                           |
| --------- | ------------------------------ |
| Student 1 | Solve using Method A (ARIMA)   |
| Student 2 | Solve using Method B (SARIMAX) |
| Student 3 | Solve using Method C (Prophet) |


The Executor Agent is like the teacher's assistant:

Gives the problem to each student (runs each model).
Collects each student's answer (metrics).
Writes all the results in one notebook (results).
Gives that notebook to the next teacher (Evaluation Agent) to decide which student performed best.

So, the Executor Agent's main responsibility is to run the selected forecasting models and collect their performance results.

| Agent             | Purpose                                                                                          | Input                                | Output                                        |
| ----------------- | ------------------------------------------------------------------------------------------------ | ------------------------------------ | --------------------------------------------- |
| **Checker Agent** | Reviews the evaluation results of all forecasting models and displays them in a formatted table. | Model evaluation results (`results`) | Rich table showing Model, MAE, RMSE, and MAPE |


=========

| Code                                          | Purpose                                           | Overall Use                                                             |
| --------------------------------------------- | ------------------------------------------------- | ----------------------------------------------------------------------- |
| `def checker_agent(state: ForecastState):`    | Defines the Checker Agent                         | Verifies and displays model evaluation results                          |
| `console.print(...)`                          | Displays "Checker Agent"                          | Indicates that the checking stage has started                           |
| `results_df = pd.DataFrame(state["results"])` | Converts the results list into a Pandas DataFrame | Makes results easy to display and analyze                               |
| `table = Table(...)`                          | Creates a Rich table                              | Displays results in a professional format                               |
| `title="Model Evaluation Results"`            | Sets the table title                              | Makes the output easy to understand                                     |
| `box=box.DOUBLE_EDGE`                         | Uses a double-line border                         | Improves the table's appearance                                         |
| `for col in results_df.columns:`              | Loops through each column                         | Adds column headers dynamically                                         |
| `table.add_column(col)`                       | Adds a column to the Rich table                   | Displays Model, MAE, RMSE, MAPE, etc.                                   |
| `for _, row in results_df.iterrows():`        | Loops through each result row                     | Reads one model's metrics at a time                                     |
| `table.add_row(*[str(i) for i in row])`       | Adds the row to the table                         | Displays each model's evaluation metrics                                |
| `console.print(table)`                        | Prints the completed table                        | Shows all model results together                                        |
| `return {}`                                   | Returns no new state                              | This agent only displays results and does not modify the workflow state |


=========



    Executor Agent
          │
          ▼
    Results List
          │
          ▼
    Convert to DataFrame
          │
          ▼
    Create Rich Table
          │
          ▼
    Add Columns
          │
          ▼
    Add Rows
          │
          ▼
    Display Evaluation Results

| Without Checker Agent              | With Checker Agent                     |
| ---------------------------------- | -------------------------------------- |
| Results remain as raw dictionaries | Results are displayed in a clear table |
| Harder to compare models           | Easy side-by-side comparison           |
| Manual inspection required         | Automatic visualization                |
| Less user-friendly                 | Professional presentation              |


====



      Role in the Multi-Agent Workflow

      Planner Agent
            │
            ▼
      Executor Agent
            │
            ▼
      Checker Agent
            │
            ▼
      Best Model Agent

=====

| Agent              | Main Responsibility                                                         |
| ------------------ | --------------------------------------------------------------------------- |
| **Planner Agent**  | Chooses which forecasting models to run.                                    |
| **Executor Agent** | Runs each forecasting model and calculates evaluation metrics.              |
| **Checker Agent**  | Displays all model evaluation results in a well-formatted comparison table. |


In [26]:

# =========================================================
# AGENT 3 → CHECKER
# =========================================================

def checker_agent(state: ForecastState):

    console.print(
        "\n[bold green]Checker Agent[/bold green]"
    )

    results_df = pd.DataFrame(state["results"])

    table = Table(
        title="Model Evaluation Results",
        box=box.DOUBLE_EDGE
    )

    for col in results_df.columns:
        table.add_column(col)

    for _, row in results_df.iterrows():

        table.add_row(
            *[str(i) for i in row]
        )

    console.print(table)

    return {}


In [27]:
# =========================================================
# AGENT 4 → DECISION
# =========================================================

def decision_agent(state: ForecastState):

    console.print(
        "\n[bold magenta]Decision Agent[/bold magenta]"
    )

    results_df = pd.DataFrame(state["results"])

    best_row = results_df.sort_values(
        "RMSE"
    ).iloc[0]

    best_model = best_row["Model"]

    console.print(
        f"Best Model → {best_model}"
    )

    return {
        "best_model": best_model
    }



| Agent              | Purpose                                                                                               | Input                                | Output                                |
| ------------------ | ----------------------------------------------------------------------------------------------------- | ------------------------------------ | ------------------------------------- |
| **Decision Agent** | Compares all forecasting models and selects the best-performing model based on the lowest RMSE value. | Model evaluation results (`results`) | Best forecasting model (`best_model`) |


======

| Code                                          | Purpose                                    | Overall Use                                |
| --------------------------------------------- | ------------------------------------------ | ------------------------------------------ |
| `def decision_agent(state: ForecastState):`   | Defines the Decision Agent                 | Chooses the best forecasting model         |
| `console.print(...)`                          | Displays "Decision Agent"                  | Indicates that model selection has started |
| `results_df = pd.DataFrame(state["results"])` | Converts the results list into a DataFrame | Makes comparison easier                    |
| `results_df.sort_values("RMSE")`              | Sorts models by RMSE in ascending order    | Places the most accurate model first       |
| `.iloc[0]`                                    | Selects the first row after sorting        | Retrieves the model with the lowest RMSE   |
| `best_model = best_row["Model"]`              | Extracts the model name                    | Identifies the best forecasting model      |
| `console.print(...)`                          | Displays the selected best model           | Shows the final decision                   |
| `return {"best_model": best_model}`           | Stores the best model in `ForecastState`   | Passes it to the next agent                |


=========



      Executor Agent
            │
            ▼
      Model Results
            │
            ▼
      Decision Agent
            │
            ▼
      Convert Results to DataFrame
            │
            ▼
      Sort by RMSE
            │
            ▼
      Select Lowest RMSE
            │
            ▼
      Store Best Model

        Planner Agent
              │
              ▼
        Executor Agent
              │
              ▼
        Checker Agent
              │
              ▼
        Decision Agent
              │
              ▼
        Forecast Agent

The Decision Agent does not train models. Its responsibility is to compare the evaluation results and choose the best-performing model.

| Agent              | Main Responsibility                                                                         |
| ------------------ | ------------------------------------------------------------------------------------------- |
| **Planner Agent**  | Chooses which forecasting models to run.                                                    |
| **Executor Agent** | Runs each forecasting model and calculates MAE, RMSE, and MAPE.                             |
| **Checker Agent**  | Displays all model evaluation results in a comparison table.                                |
| **Decision Agent** | Compares the models and selects the one with the lowest RMSE as the best forecasting model. |


In [28]:
# =========================================================
# AGENT 5 → TUNING
# =========================================================

def tuning_agent(state: ForecastState):

    console.print(
        "\n[bold blue]Tuning Agent[/bold blue]"
    )

    param_grid = {
        "p": [0, 1, 2],
        "d": [0, 1],
        "q": [0, 1, 2]
    }

    best_score = np.inf

    best_params = None

    for params in ParameterGrid(param_grid):

        try:

            model = ARIMA(
                y_train,
                order=(
                    params["p"],
                    params["d"],
                    params["q"]
                )
            ).fit()

            pred = model.forecast(len(y_test))

            rmse = evaluate(
                y_test,
                pred
            )["RMSE"]

            if rmse < best_score:

                best_score = rmse

                best_params = params

        except:
            pass


    auto_model = auto_arima(
        y_train,
        seasonal=False
    )

    auto_pred = auto_model.predict(
        n_periods=len(y_test)
    )

    auto_rmse = evaluate(
        y_test,
        auto_pred
    )["RMSE"]

    scores = {
        "GridSearch": best_score,
        "AutoARIMA": auto_rmse
    }

    best_method = min(
        scores,
        key=scores.get
    )

    console.print(
        f"Best Tuning Method → {best_method}"
    )

    console.print(
        f"Best Params → {best_params}"
    )

    return {
        "tuning_method": best_method,
        "best_params": best_params
    }



| Agent            | Purpose                                                                                                           | Input                                              | Output                                 |
| ---------------- | ----------------------------------------------------------------------------------------------------------------- | -------------------------------------------------- | -------------------------------------- |
| **Tuning Agent** | Finds the best forecasting parameters and the best tuning method by comparing **Grid Search** and **Auto ARIMA**. | Training data (`y_train`), Testing data (`y_test`) | Best tuning method and best parameters |


=======

| Code                                      | Purpose                                      | Overall Use                                           |
| ----------------------------------------- | -------------------------------------------- | ----------------------------------------------------- |
| `def tuning_agent(state: ForecastState):` | Defines the Tuning Agent                     | Starts the hyperparameter tuning process              |
| `console.print(...)`                      | Displays "Tuning Agent"                      | Indicates tuning has started                          |
| `param_grid = {...}`                      | Defines possible values of `p`, `d`, and `q` | Creates combinations for Grid Search                  |
| `best_score = np.inf`                     | Initializes RMSE with infinity               | Ensures the first valid RMSE becomes the current best |
| `best_params = None`                      | Initializes best parameters                  | Stores the best ARIMA configuration                   |


=========

| Code                        | Purpose                                         | Output                                |
| --------------------------- | ----------------------------------------------- | ------------------------------------- |
| `ParameterGrid(param_grid)` | Generates all combinations of `p`, `d`, and `q` | Multiple ARIMA parameter sets         |
| `ARIMA(...).fit()`          | Trains an ARIMA model with one parameter set    | Trained model                         |
| `forecast()`                | Predicts values for the test period             | Predictions                           |
| `evaluate()`                | Calculates RMSE                                 | Model accuracy                        |
| `if rmse < best_score`      | Checks if the current model is better           | Updates the best score and parameters |


        Training Data
              │
              ▼
        Tuning Agent
              │
              ├───────────────┐
              │               │
              ▼               ▼
        Grid Search      Auto ARIMA
              │               │
              ▼               ▼
        Calculate RMSE   Calculate RMSE
              │               │
              └──────┬────────┘
                    ▼
        Compare RMSE
                    ▼
        Select Best Method
                    ▼
        Return Best Parameters

        Planner Agent
              │
              ▼
        Executor Agent
              │
              ▼
        Checker Agent
              │
              ▼
        Decision Agent
              │
              ▼
        Tuning Agent
              │
              ▼
        Forecast Agent


        The Tuning Agent improves the selected model by searching for the best hyperparameters before the final forecasting step.

| Agent              | Responsibility                                                                                                                    |
| ------------------ | --------------------------------------------------------------------------------------------------------------------------------- |
| **Planner Agent**  | Chooses which forecasting models to run.                                                                                          |
| **Executor Agent** | Trains the selected models and computes MAE, RMSE, and MAPE.                                                                      |
| **Checker Agent**  | Displays the evaluation results in a comparison table.                                                                            |
| **Decision Agent** | Selects the best forecasting model based on the lowest RMSE.                                                                      |
| **Tuning Agent**   | Optimizes the selected model by comparing **Grid Search** and **Auto ARIMA**, then returns the best tuning method and parameters. |


In [29]:
# =========================================================
# AGENT 6 → FORECASTING
# =========================================================

def forecasting_agent(state: ForecastState):

    console.print(
        "\n[bold red]Forecasting Agent[/bold red]"
    )

    best_model = state["best_model"]

    y_full = df[TARGET_COLUMN]

    if best_model == "ARIMA":

        model = ARIMA(
            y_full,
            order=(1, 1, 1)
        )

        fit = model.fit()

        forecast = fit.forecast(
            steps=FORECAST_DAYS
        )


    elif best_model == "SARIMAX":

        model = SARIMAX(
            y_full,
            order=(1, 1, 1)
        )

        fit = model.fit()

        forecast = fit.forecast(
            steps=FORECAST_DAYS
        )


    elif best_model == "Prophet":

        prophet_df = df.reset_index()[
            ["date", TARGET_COLUMN]
        ]

        prophet_df.columns = ["ds", "y"]

        model = Prophet()

        model.fit(prophet_df)

        future = model.make_future_dataframe(
            periods=FORECAST_DAYS
        )

        forecast_result = model.predict(future)

        forecast = forecast_result["yhat"].tail(
            FORECAST_DAYS
        ).values


    future_dates = pd.date_range(
        start=df.index[-1] + pd.Timedelta(days=1),
        periods=FORECAST_DAYS
    )

    forecast_df = pd.DataFrame({

        "Date": future_dates,

        "Predicted_Sales": forecast
    })

    console.print(
        Panel(
            str(forecast_df),
            title="Next 7 Days Forecast"
        )
    )

    return {
        "forecast_df": forecast_df
    }



In [30]:
print(checker_agent)

<function checker_agent at 0x7c47bfd80c20>


In [31]:
# =========================================================
# BUILD GRAPH
# =========================================================

builder = StateGraph(ForecastState)

builder.add_node(
    "Planner",
    planner_agent
)

builder.add_node(
    "Executor",
    executor_agent
)

builder.add_node(
    "Checker",
    checker_agent
)

builder.add_node(
    "Decision",
    decision_agent
)

builder.add_node(
    "Tuning",
    tuning_agent
)

builder.add_node(
    "Forecast",
    forecasting_agent
)



In [32]:

# =========================================================
# FLOW
# =========================================================

builder.set_entry_point("Planner")

builder.add_edge(
    "Planner",
    "Executor"
)

builder.add_edge(
    "Executor",
    "Checker"
)

builder.add_edge(
    "Checker",
    "Decision"
)

builder.add_edge(
    "Decision",
    "Tuning"
)

builder.add_edge(
    "Tuning",
    "Forecast"
)

builder.add_edge(
    "Forecast",
    END
)



In [33]:
# =========================================================
# COMPILE GRAPH
# =========================================================

graph = builder.compile()


# =========================================================
# VISUALIZE GRAPH
# =========================================================

try:

    print(
        graph.get_graph().draw_ascii()
    )

except:
    pass


In [34]:
# =========================================================
# RUN GRAPH
# =========================================================

final_state = graph.invoke({

    "models": [],

    "results": [],

    "best_model": "",

    "tuning_method": "",

    "best_params": {},

    "forecast_df": None,

    "error": None
})

Planner Agent Running

Selected Models → ['ARIMA', 'SARIMAX', 'Prophet']

Executor Agent Running

ARIMA Completed

SARIMAX Completed

...

Best Tuning Method → GridSearch

Best Params → {'d': 1, 'p': 1, 'q': 2}

Forecasting Agent

INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
INFO:prophet:n_changepoints greater than number of observations. Using 17.


╭───────────────────────────────────────────── Next 7 Days Forecast ──────────────────────────────────────────────╮
│         Date  Predicted_Sales                                                                                   │
│ 0 2024-01-31        70.565730                                                                                   │
│ 1 2024-02-01        74.236068                                                                                   │
│ 2 2024-02-02        77.727238                                                                                   │
│ 3 2024-02-03        71.728111                                                                                   │
│ 4 2024-02-04        77.568043                                                                                   │
│ 5 2024-02-05        76.753352                                                                                   │
│ 6 2024-02-06        77.252589                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

| Step | Agent                 | Responsibility                                                      | Output                                                |
| ---- | --------------------- | ------------------------------------------------------------------- | ----------------------------------------------------- |
| 1    | **Planner Agent**     | Reads the dataset and decides which forecasting models to evaluate. | Selected **ARIMA**, **SARIMAX**, and **Prophet**.     |
| 2    | **Executor Agent**    | Trains each selected forecasting model.                             | Successfully trained ARIMA, SARIMAX, and Prophet.     |
| 3    | **Checker Agent**     | Evaluates all trained models using error metrics.                   | Generated MAE, RMSE, and MAPE comparison table.       |
| 4    | **Decision Agent**    | Chooses the best-performing model.                                  | Selected **Prophet** because it had the lowest error. |
| 5    | **Tuning Agent**      | Finds the best hyperparameters.                                     | Used Grid Search and found **p=1, d=1, q=2**.         |
| 6    | **Forecasting Agent** | Uses the chosen model to predict future values.                     | Produced the next 7-day sales forecast.               |


                Historical Sales Data
                        │
                        ▼
               Planner Agent
                        │
                        ▼
      Select Models (ARIMA, SARIMAX, Prophet)
                        │
                        ▼
              Executor Agent
                        │
         ┌──────────┬──────────┬──────────┐
         ▼          ▼          ▼
      ARIMA      SARIMAX    Prophet
         │          │          │
         └──────────┴──────────┘
                    ▼
             Checker Agent
      Compare MAE, RMSE, MAPE
                    ▼
            Decision Agent
         Select Best Model
                    ▼
             Tuning Agent
     Optimize ARIMA Parameters
                    ▼
          Forecasting Agent
       Predict Next 7 Days